In [1]:
import pandas as pd
import sys
import os
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Add the EDP directory to the Python path
sys.path.append(os.path.abspath(os.path.join('..', 'EDP')))

/Users/suhaibbasir/Documents/CS/MSc/Thesis/.conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from parser_2 import Parser2

In [6]:
xml_file_path = '/Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/output_solr34.xml'

data = Parser2.XLMtoString(xml_file_path)

In [7]:
display(data[:10])

[{'id': 'ID: /2021672/resource_document_mauritshuis_397',
  'text': 'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_340',
  'text': 'The Annunciation by Francesco Solimena, dated 1693 - 1693'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_426',
  'text': 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_432',
  'text': 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_354',
  'text': 'St Barbara by Parmigianino, dated None'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_181',
  'text': "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737"},
 {'id': 'ID: /2021672/resource_document_mauritshuis_195',
  'text': 'A Fishmo

In [12]:
tokenizer = AutoTokenizer.from_pretrained("naver/splade-cocondenser-ensembledistil")
model = AutoModelForMaskedLM.from_pretrained("naver/splade-cocondenser-ensembledistil")

In [8]:
data[0]["text"]

'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'

In [9]:
import torch 

def get_max_logits(output, tokens):
    return torch.max(
        torch.log(
            1 + torch.relu(output.logits)
        ) * tokens.attention_mask.unsqueeze(-1),
        dim=1)[0].squeeze().detach().cpu().numpy()

In [10]:
def builder(records: list):
    ids = [x['id'] for x in records]
    text = [x['text'] for x in records]
    # create sparse vecs
    tokens = tokenizer(
        text, return_tensors='pt',
        padding=True, truncation=True
    )
    sparse_vecs = get_max_logits(model(**tokens), tokens)
    upserts = []
    for _id, sparse_vec, text in zip(ids, sparse_vecs, text):
        upserts.append({
            'id': _id,
            'sparse_values': sparse_vec,
            'text': text
        })
    return upserts

In [13]:
builder(data[:2])

[{'id': 'ID: /2021672/resource_document_mauritshuis_397',
  'sparse_values': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
  'text': 'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_340',
  'sparse_values': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
  'text': 'The Annunciation by Francesco Solimena, dated 1693 - 1693'}]

In [14]:
from milvus import default_server

from pymilvus import FieldSchema, CollectionSchema, DataType, Collection, utility, connections

In [16]:
default_server.start()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [38]:
connections.connect(
    host = '127.0.0.1', 
    port = default_server.listen_port,
    max_message_size = 64 * 1024 * 1024
)

print(utility.get_server_version())

v2.3.8-2-g99b8bbd82-lite


In [19]:
upserts = builder(data)

In [39]:
ids = [x['id'] for x in upserts]
sparse_embeddings = [x['sparse_values'] for x in upserts]
text = [x['text'] for x in upserts]

In [40]:
# find max lenght of an embedding value in embeddings
max_len = max([len(x) for x in sparse_embeddings])
print(max_len)

30522


In [41]:
# Define the schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, max_length=50, is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=max_len), 
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535)
]

schema = CollectionSchema(fields, "Schema for europeana mauritshuis data")

# Create the collection if it doesn't exist
collection_name = "eana_mauritshuis"
if not utility.has_collection(collection_name):
    collection = Collection(name=collection_name, schema=schema)
    print(f"Collection {collection_name} created.")
else:
    collection = Collection(name=collection_name)
    collection.drop()
    collection = Collection(name=collection_name, schema=schema)
    print(f"Collection {collection_name} already exists. but dropped and recreated.")


Collection eana_mauritshuis already exists. but dropped and recreated.


In [42]:
print(collection.schema)
print(collection.name)

{'auto_id': False, 'description': 'Schema for europeana mauritshuis data', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 50}, 'is_primary': True, 'auto_id': False}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 30522}}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}], 'enable_dynamic_field': False}
eana_mauritshuis


In [43]:
milvus_data = [
    ids,
    sparse_embeddings,
    text
]
print((milvus_data[1][0]))
len(milvus_data[1][0])

[0. 0. 0. ... 0. 0. 0.]


30522

In [53]:
milvus_data_first_half = [
    ids[:len(ids)//2],
    sparse_embeddings[:len(ids)//2],
    text[:len(ids)//2]
]

milvus_data_second_half = [
    ids[len(ids)//2:],
    sparse_embeddings[len(ids)//2:],
    text[len(ids)//2:]
]

In [54]:
collection.insert(milvus_data_first_half)

(insert count: 415, delete count: 0, upsert count: 0, timestamp: 450391022307901443, success count: 415, err count: 0, cost: 0)

In [55]:
collection.insert(milvus_data_second_half)

(insert count: 416, delete count: 0, upsert count: 0, timestamp: 450391026174525442, success count: 416, err count: 0, cost: 0)

In [56]:
index_params = {
    "metric_type": "COSINE",
    "index_type": "IVF_FLAT",
    "params": {"nlist": 1024}
}

collection.create_index(field_name="embedding", index_params=index_params)

Status(code=0, message=)

In [57]:
collection.load()


In [85]:

query = input("Enter a query: ")

query_tokens = tokenizer(query, return_tensors="pt")
query_output = model(**query_tokens)

query_sparse_emb = get_max_logits(query_output, query_tokens)

search_params = {"metric_type": "COSINE", "params": {"nprobe": 10}}

results = collection.search(
    data=[query_sparse_emb],
    anns_field="embedding",
    param=search_params,
    limit=10,
    output_fields=["id", "text"],
)

print("query: ", query)
for result in results[0]:
    print(f"Document ID: {result.id}, Text: {result.entity.get('text')}, Distance: {result.distance}")



query:  wearing jewelry
Document ID: ID: /2021672/resource_document_mauritshuis_48, Text: Sumptuous Fruit Still Life with Jewellery Box by Jan Davidsz de Heem, dated 1655 - 1650, Distance: 0.17938175797462463
Document ID: ID: /2021672/resource_document_mauritshuis_59, Text: The Raven Robbed of the Feathers He Wore to Adorn Himself by Melchior d' Hondecoeter, dated 1671 - 1671, Distance: 0.11085028946399689
Document ID: ID: /2021672/resource_document_mauritshuis_1163, Text: Man in Oriental Dress by Frans van Mieris the Elder, dated 1665 - 1665, Distance: 0.09126143157482147
Document ID: ID: /2021672/resource_document_mauritshuis_670, Text: Girl with a Pearl Earring by Johannes Vermeer, dated 1665 - 1665, Distance: 0.07628100365400314
Document ID: ID: /2021672/resource_document_mauritshuis_1212, Text: Narcissi, Periwinkle and Violets in a Ewer by Ludger tom Ring the Younger, dated 1562 - 1562, Distance: 0.06862012296915054
Document ID: ID: /2021672/resource_document_mauritshuis_149, Text